# LLM Adaptation Workflow
### Notebook 00 — Project Overview

---

## What are we building?

Large Language Models (LLMs) like GPT-4 or Llama are trained on massive amounts of internet text. They know a lot — but they don't know **your** company's documents, your clients, your internal processes, or your domain-specific terminology.

This project builds a **complete pipeline** to take a general-purpose open-source model and adapt it to a specific knowledge domain — in our case, financial services using public SEC filings.

The workflow is designed to be **domain-agnostic**: once you understand each step, you can swap the financial corpus for legal documents, medical records, engineering runbooks, or any other proprietary knowledge base.

---

## The big picture

There are two main ways to make a model "know" your domain:

| Approach | What it does | When to use it |
|----------|-------------|----------------|
| **RAG** (Retrieval-Augmented Generation) | At query time, retrieve relevant documents and give them to the model as context | Knowledge changes frequently, large corpora, fast setup |
| **Fine-tuning** | Retrain the model's weights on your data | Consistent tone/style, specialised reasoning, faster inference |

We build **both** and compare them. In practice, many production systems combine the two.

---

## Workflow diagram

```
┌─────────────────────────────────────────────────────────┐
│               Raw Documents / Public Corpus              │
│          (SEC filings, PDFs, web pages, reports)         │
└────────────────────────┬────────────────────────────────┘
                         │
                         ▼
              ┌──────────────────┐
              │ 02 · Data Prep   │  Clean, chunk, format into
              │                  │  instruction datasets
              └────────┬─────────┘
                       │
              ┌────────┴─────────┐
              │                  │
              ▼                  ▼
   ┌──────────────────┐  ┌──────────────────┐
   │  03 · RAG        │  │  04 · Fine-Tune  │  Two paths —
   │  Vector store +  │  │  LoRA adapters   │  use one or both
   │  retrieval       │  │  on top of LLM   │
   └────────┬─────────┘  └────────┬─────────┘
            │                     │
            └──────────┬──────────┘
                       │
                       ▼
              ┌──────────────────┐
              │  05 · Alignment  │  Teach the model what
              │  DPO / Prefs     │  "good" answers look like
              └────────┬─────────┘
                       │
                       ▼
              ┌──────────────────┐
              │  06 · Evaluate   │  Measure quality before
              │  Benchmark       │  and after each step
              └──────────────────┘
```

---

## Notebooks in order

| # | Notebook | What you'll learn |
|---|----------|-------------------|
| 00 | **Project Overview** (this one) | Architecture, decisions, setup |
| 01 | **Foundation Models** | What LLMs are, how transformers work, loading models |
| 02 | **Data Preparation** | Building instruction datasets from raw documents |
| 03 | **RAG Pipeline** | Embeddings, vector stores, retrieval |
| 04 | **Fine-Tuning** | LoRA — training without needing a data centre |
| 05 | **Alignment** | DPO — teaching the model what "good" means |
| 06 | **Evaluation** | Measuring and comparing model quality |

---

## How to adapt this to your domain

Each notebook has an **"🔧 Adapt This"** section. The main things you'd change:

1. **Data source** (Notebook 02) — replace SEC filings with your documents
2. **System prompt** (`utils/templates.py`) — describe your domain and task
3. **Evaluation questions** (Notebook 06) — use questions relevant to your domain

Everything else — the training loop, the RAG architecture, the alignment step — stays the same.

---

## Hardware note

All notebooks are designed for **Apple Silicon Macs** (M1/M2/M3/M4) using PyTorch's **MPS backend**.

If you're running on a different machine:
- **NVIDIA GPU**: change `device = "mps"` to `device = "cuda"`
- **CPU only**: change to `device = "cpu"` (will be slow but works for small models)
- **Google Colab**: change to `device = "cuda"` and enable T4 GPU in runtime settings

In [ ]:
# Let's start by checking your environment is set up correctly.
# Run this cell first to confirm everything is installed and your hardware is detected.

import sys
import torch

print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")

# Detect the best available device
if torch.backends.mps.is_available():
    device = "mps"
    print("Device         : Apple Silicon MPS ✓")
elif torch.cuda.is_available():
    device = "cuda"
    print(f"Device         : CUDA — {torch.cuda.get_device_name(0)} ✓")
else:
    device = "cpu"
    print("Device         : CPU (no accelerator found)")

print(f"\nUsing device   : {device}")

In [ ]:
# Check that the key libraries are installed

libraries = [
    "transformers",
    "datasets",
    "peft",
    "trl",
    "sentence_transformers",
    "faiss",
    "evaluate",
]

print("Checking required libraries...\n")
all_ok = True
for lib in libraries:
    try:
        __import__(lib)
        print(f"  ✓ {lib}")
    except ImportError:
        print(f"  ✗ {lib}  ← run: pip install {lib}")
        all_ok = False

print()
if all_ok:
    print("All libraries found. You're ready to go!")
else:
    print("Some libraries are missing. Run: pip install -r requirements.txt")

---

## Key concepts — a quick glossary

These terms come up throughout the project. Don't worry if they're new — each notebook explains them in context.

| Term | Plain English |
|------|---------------|
| **Foundation model** | A large pre-trained model (e.g. Llama, Phi) trained on broad internet text |
| **Fine-tuning** | Continuing to train a model on your specific data |
| **LoRA** | A trick to fine-tune only a small fraction of the model's parameters |
| **Instruction tuning** | Training the model to follow instructions in a Q&A format |
| **RAG** | Giving the model relevant documents at query time instead of baking knowledge into weights |
| **Embeddings** | Turning text into a list of numbers that captures semantic meaning |
| **Vector store** | A database that can find semantically similar text by comparing embeddings |
| **DPO** | A way to train the model to prefer certain responses over others |
| **Alignment** | Making the model behave in ways that are helpful, honest, and appropriate |
| **Hallucination** | When a model generates plausible-sounding but incorrect information |
| **Benchmark** | A standardised test used to measure and compare model quality |

---

▶ **Next: [Notebook 01 — Foundation Models](01_foundation_models.ipynb)**